In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

In [0]:
# Correct URL if necessary
df1 = pd.read_csv("https://github.com/davidtaki/Online_Retail_Project/raw/main/online_retail_utf_8_2009_2010.csv", 
                 sep=';', 
                 decimal=',')

# Save the file locally
df1.to_csv("online_retail_fixed.csv", 
          sep=';', 
          index=False, 
          decimal='.')

In [0]:
df2 = pd.read_csv("https://github.com/davidtaki/Online_Retail_Project/raw/main/online_retail_utf_8_2010_2011.csv", 
                 sep=';', 
                 decimal=',')
df2.to_csv("online_retail_fixed_2.csv", 
          sep=';', 
          index=False, 
          decimal='.')

In [0]:
df_appended = pd.concat([df1, df2], ignore_index=True)
df_appended.to_csv("online_retail_fixed_appended.csv", 
          sep=';', 
          index=False, 
          decimal='.')

In [0]:
df_appended.info()

In [0]:
df_appended.head()

In [0]:
df_appended.describe()

In [0]:
df_appended['InvoiceDate'] = pd.to_datetime(df_appended['InvoiceDate'], 
                                            format='%Y-%m-%d')

### CREATE DATABASE



In [0]:
%sql
CREATE OR REPLACE TABLE online_retail.sales_raw (
  Invoice        STRING,
  StockCode      STRING,
  Description    STRING,
  Quantity       INT,
  InvoiceDate    DATE,    
  UnitPrice      DOUBLE,
  CustomerID     DOUBLE,
  Country        STRING
)
COMMENT 'Bronze: raw CSV snapshot (UC-managed storage)';

In [0]:
df_fixed = (df_appended_spark
    .withColumnRenamed("Customer ID", "customer_id")
    .withColumn("Quantity", F.col("Quantity").cast("int"))
    .withColumn("Price",     F.col("Price").cast("decimal(10,2)"))
    .withColumn(
        "InvoiceDate",
        F.coalesce(
            F.to_timestamp("InvoiceDate", "yyyy.MM.dd H:mm"),   # 2009.12.01 7:45
            F.to_timestamp("InvoiceDate", "yyyy.MM.dd HH:mm"),  # 2009.12.01 07:45
            F.to_timestamp("InvoiceDate", "yyyy-MM-dd HH:mm:ss")# fall-back if any
        )
    )
)

In [0]:
(df_fixed
  .write
  .format("delta")
  .mode("overwrite")          # or "append" if you’re adding rows
  .option("overwriteSchema", "true")   # keep life simple if you overwrite
  .saveAsTable("online_retail.sales_raw")
)

In [0]:
%sql
-- Silver layer
CREATE TABLE online_retail.sales_silver
USING DELTA
PARTITIONED BY (invoice_year, invoice_month)
AS
SELECT
  Invoice,
  StockCode,
  coalesce(Description, 'Unknown')              AS Description,
  Quantity                                 AS Quantity,   
  to_timestamp(InvoiceDate, 'yyyy.MM.dd H:mm')  AS InvoiceTS,
  year(to_timestamp(InvoiceDate, 'yyyy.MM.dd H:mm'))  AS invoice_year,
  month(to_timestamp(InvoiceDate, 'yyyy.MM.dd H:mm')) AS invoice_month,
  Price,
  customer_id AS CustomerID,
  Country
FROM online_retail.sales_raw
WHERE Price > 0;

##  Create a hierarchie level table for articles

In [0]:
silver_df = spark.table("online_retail.sales_silver")

article_df = (silver_df
              .select("StockCode", "Description")
              .distinct())

In [0]:
pandas_df = article_df.toPandas()
display(pandas_df)

In [0]:
pandas_df.to_csv("stockcode_db.csv", 
          sep=';', 
          index=False, 
          decimal='.')

In [0]:
%sql
CREATE OR REPLACE TABLE online_retail.sales_articles (
  StockCode      STRING,
  Description    STRING
)
COMMENT 'articles_table with desriptions';

In [0]:
(article_df
  .write
  .format("delta")
  .mode("overwrite")         
  .option("overwriteSchema", "true")   
  .saveAsTable("online_retail.sales_articles")
)

In [0]:
%sql
SELECT 
    a.StockCode,
    a.Description
FROM online_retail.sales_articles a  LIMIT 10;